# Extract features από MDVR-KCL και Iyer 2023 datasets

Εφαρμόζουμε ΠΡΩΤΑ το audio preprocessing pipeline (noise reduction, normalize, trim) και ΜΕΤΑ το feature extraction. Έτσι όλα τα training data έχουν συνεπή κανονικοποίηση που θα ταιριάζει με το live audio από browser.

In [1]:
import sys, re
sys.path.insert(0, '..')

import pandas as pd
import tempfile
from pathlib import Path
from tqdm import tqdm

from src.features import extract_features, FEATURE_NAMES
from src.preprocessing import preprocess_audio


def extract_with_preprocessing(wav_path):
    """Preprocess + extract features. Επιστρέφει dict."""
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        preprocess_audio(wav_path, output_path=tmp.name)
        return extract_features(tmp.name)

## MDVR-KCL extraction

In [2]:
MDVR_ROOT = Path('../data/mdvr_kcl/26-29_09_2017_KCL')
mdvr_files = list(MDVR_ROOT.rglob('*.wav'))
print(f'MDVR files: {len(mdvr_files)}')

def mdvr_parse(wav_path):
    parts = [p.lower() for p in wav_path.parts]
    label = 1 if 'pd' in parts else (0 if 'hc' in parts else -1)
    m = re.match(r'(ID\d+_(?:pd|hc))', wav_path.name, re.IGNORECASE)
    subject = m.group(1) if m else wav_path.stem
    return label, subject

mdvr_rows = []
for wav in tqdm(mdvr_files, desc='MDVR'):
    label, subject = mdvr_parse(wav)
    if label == -1: continue
    try:
        feats = extract_with_preprocessing(wav)
        feats['class'] = label
        feats['subject'] = subject
        feats['filename'] = wav.name
        mdvr_rows.append(feats)
    except Exception as e:
        print(f'  Error {wav.name}: {e}')

mdvr_df = pd.DataFrame(mdvr_rows)
mdvr_df.to_csv('../data/mdvr_kcl/mdvr_features.csv', index=False)
print(f'\nSaved {mdvr_df.shape[0]} rows. Classes: {mdvr_df["class"].value_counts().to_dict()}')

MDVR files: 73


MDVR:   0%|          | 0/73 [00:00<?, ?it/s]

MDVR:   1%|▏         | 1/73 [00:02<03:34,  2.98s/it]

MDVR:   3%|▎         | 2/73 [00:05<03:12,  2.71s/it]

MDVR:   4%|▍         | 3/73 [00:08<03:05,  2.66s/it]

...

MDVR:  97%|█████████▋| 71/73 [02:36<00:04,  2.08s/it]

MDVR:  99%|█████████▊| 72/73 [02:38<00:02,  2.14s/it]

MDVR: 100%|██████████| 73/73 [02:40<00:00,  2.08s/it]

MDVR: 100%|██████████| 73/73 [02:40<00:00,  2.20s/it]


Saved 73 rows. Classes: {0: 42, 1: 31}


## Iyer 2023 extraction

Sustained /a/ recordings, 41 HC + 40 PD. Ίδιο task με το UCI.

In [3]:
IYER_ROOT = Path('../data/iyer')
iyer_hc = list((IYER_ROOT / 'HC_AH').glob('*.wav'))
iyer_pd = list((IYER_ROOT / 'PD_AH').glob('*.wav'))
print(f'Iyer HC: {len(iyer_hc)}, PD: {len(iyer_pd)}')

Iyer HC: 41, PD: 40


In [4]:
iyer_rows = []
for wav in tqdm(iyer_hc, desc='Iyer HC'):
    try:
        feats = extract_with_preprocessing(wav)
        feats['class'] = 0
        feats['subject'] = wav.stem  # κάθε file = ξεχωριστό subject (αν δούμε αργότερα ότι όχι, fix)
        feats['filename'] = wav.name
        iyer_rows.append(feats)
    except Exception as e:
        print(f'  Error {wav.name}: {e}')

for wav in tqdm(iyer_pd, desc='Iyer PD'):
    try:
        feats = extract_with_preprocessing(wav)
        feats['class'] = 1
        feats['subject'] = wav.stem
        feats['filename'] = wav.name
        iyer_rows.append(feats)
    except Exception as e:
        print(f'  Error {wav.name}: {e}')

iyer_df = pd.DataFrame(iyer_rows)
iyer_df.to_csv('../data/iyer/iyer_features.csv', index=False)
print(f'\nSaved {iyer_df.shape[0]} rows. Classes: {iyer_df["class"].value_counts().to_dict()}')

Iyer HC:   0%|          | 0/41 [00:00<?, ?it/s]

Iyer HC:   5%|▍         | 2/41 [00:00<00:02, 19.12it/s]

Iyer HC:  10%|▉         | 4/41 [00:00<00:02, 17.97it/s]

Iyer HC:  15%|█▍        | 6/41 [00:00<00:02, 17.33it/s]

Iyer HC:  20%|█▉        | 8/41 [00:00<00:02, 16.40it/s]

...

Iyer PD:  85%|████████▌ | 34/40 [00:02<00:00, 12.76it/s]

Iyer PD:  90%|█████████ | 36/40 [00:02<00:00, 12.28it/s]

Iyer PD:  95%|█████████▌| 38/40 [00:02<00:00, 13.07it/s]

Iyer PD: 100%|██████████| 40/40 [00:03<00:00, 13.14it/s]


Saved 81 rows. Classes: {0: 41, 1: 40}
